# GlacierGuards: lake segmentation training on Kaggle
GPU T4/P100, Internet ON for the first run (pip + ImageNet weights). Attach the `glacierguards-gloid` dataset as input.

In [ ]:
import glob, os, shutil, subprocess, json
DATA = os.path.dirname(glob.glob('/kaggle/input/**/splits.json', recursive=True)[0])
shutil.copytree(f'{DATA}/code', '/kaggle/working/code', dirs_exist_ok=True)
os.chdir('/kaggle/working/code')
subprocess.run('pip -q install segmentation-models-pytorch rasterio pyproj pyyaml', shell=True)
import torch; print(DATA, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print(json.load(open(f'{DATA}/splits.json'))['test_lakes'])

In [ ]:
COMMON = f'--data-root {DATA} --epochs 40 --batch-size 16 --lr 3e-4'
# sanity run: CV fold 0, fused channels
!python src/training/train_segmentation.py {COMMON} --split cv --fold 0

In [ ]:
# 5-fold lake-grouped CV for fused, then the two ablations (about 20 min per run on a T4)
for ch in ['fused', 'sar', 'optical']:
    for fold in range(5):
        !python src/training/train_segmentation.py {COMMON} --split cv --fold {fold} --channels {ch}

In [ ]:
# final model on the whole pool, evaluated on the held-out test lakes
!python src/training/train_segmentation.py {COMMON} --split test --channels fused

In [ ]:
# optional: leave-one-lake-out for chosen lakes (edit the list)
lakes = json.load(open(f'{DATA}/splits.json'))['lolo'][:6]
for l in lakes:
    !python src/training/train_segmentation.py {COMMON} --split lake --holdout-lake {l}

In [ ]:
import glob, pandas as pd
rows = []
for f in sorted(glob.glob('models/*_metrics.json')):
    m = json.load(open(f))
    rows.append({k: m.get(k) for k in ['name','channels','split','best_epoch','iou','f1','precision','recall','accuracy']})
    for lake, v in m.get('per_lake', {}).items():
        rows.append({'name': m['name'], 'channels': m['channels'], 'split': 'lake:' + lake, **{k: v[k] for k in ['iou','f1','precision','recall','accuracy']}})
pd.set_option('display.width', 200); display(pd.DataFrame(rows).round(3))

In [ ]:
!cd /kaggle/working && zip -qr results.zip code/models && ls -la results.zip   # download from the Output tab (the .pt files are inside)